# 🌿 HSI ViT U-Net — Tree Crown Segmentation

ViT encoder + U-Net decoder as comparison against HSI 3D CNN baseline.

**Architecture:**
```
Input (B_bands, 32, 32)
  SpectralProjection → (64, 32, 32)   # band-agnostic
  PatchEmbed 4x4 → 64 tokens dim=256
  ViT encoder depth=4, heads=8
  Reshape → (256, 8, 8)
  U-Net decoder → (1, 32, 32)
```

## 📦 Cell 1 — Imports

In [ ]:
import os, random, datetime
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import tifffile
from dotenv import load_dotenv
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'   GPU: {torch.cuda.get_device_name(0)}')

## ⚙️ Cell 2 — Configuration

In [ ]:
load_dotenv('../neon-tree-species/.env')
if wandb.run is not None:
    wandb.finish()
wandb.login(key=os.getenv('WANDB_API_KEY'))

In [ ]:


run_label = 'hsi_vit_unet'
run_id    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

CFG = {
    'preprocessed_dir' : '../data/hsi_preprocessed',
    'save_dir'         : f'../neon-tree-species/run_{run_id}_{run_label}/checkpoints',
    # HSI
    'n_bands_raw'   : 426,
    'spectral_out'  : 64,    # SpectralProjection output channels
    'img_size'      : 32,
    # ViT — 32x32 image, 4x4 patches → 64 tokens
    'patch_size'    : 4,
    'vit_depth'     : 4,
    'vit_heads'     : 8,
    'vit_hidden'    : 256,
    'vit_mlp_dim'   : 1024,
    'dropout'       : 0.1,
    # Training
    'epochs'        : 80,
    'batch_size'    : 16,
    'lr'            : 5e-4,
    'weight_decay'  : 1e-4,
    'num_workers'   : 2,
    'warmup_epochs' : 8,
    # Loss
    'focal_gamma'   : 2.0,
    'pos_weight'    : 2.0,
    'dice_weight'   : 0.5,
}

os.makedirs(CFG['save_dir'], exist_ok=True)


In [ ]:
wandb.init(project='tree-crown-segmentation',
           name=f'run_{run_id}_{run_label}', config=CFG)
print(f'wandb: {wandb.run.url}')

## 🎨 Cell 3 — HSI Dataset

In [ ]:
from PIL import Image

def get_train_transforms(img_size):
    return A.Compose([
        A.PadIfNeeded(min_height=img_size, min_width=img_size, border_mode=0, fill=0),
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ElasticTransform(alpha=60, sigma=6, p=0.3),
        A.RandomCrop(img_size, img_size, p=1.0),
        A.OneOf([
            A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=30, val_shift_limit=20),
        ], p=0.6),
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(std_range=(0.02, 0.1), p=0.2),
        A.RandomShadow(shadow_roi=(0, 0, 1, 1), p=0.2),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_val_transforms(img_size):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

class TreeCrownDataset(Dataset):
    """
    Loads paired RGB images and binary masks.
    Masks can be 0/1 or 0/255 — auto-normalized to 0/1.
    Image+mask transforms are applied together via Albumentations,
    guaranteeing identical geometric transforms for both.
    """
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self): return len(self.image_paths)

    def __getitem__(self, idx):
        image = np.array(Image.open(self.image_paths[idx]).convert('RGB'))
        mask  = np.array(Image.open(self.mask_paths[idx]).convert('L'))
        mask  = mask.astype(np.float32)
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']         # Tensor [3, H, W]
            mask  = augmented['mask']          # Tensor [H, W]
        return image, mask.unsqueeze(0)        # mask → [1, H, W]

def build_dataloaders(cfg):
    root = Path(cfg['data_root'])
    train_imgs  = sorted((root / 'train' / 'img'    / 'tree').glob('*'))
    train_masks = sorted((root / 'train' / 'labels' / 'tree').glob('*'))
    val_imgs    = sorted((root / 'val'   / 'img'    / 'tree').glob('*'))
    val_masks   = sorted((root / 'val'   / 'labels' / 'tree').glob('*'))
    assert len(train_imgs) == len(train_masks) > 0
    assert len(val_imgs)   == len(val_masks)   > 0
    train_ds = TreeCrownDataset(train_imgs, train_masks, get_train_transforms(cfg['img_size']))
    val_ds   = TreeCrownDataset(val_imgs,   val_masks,   get_val_transforms(cfg['img_size']))
    train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,
                              num_workers=cfg['num_workers'], pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False,
                              num_workers=cfg['num_workers'], pin_memory=True)
    print(f'Train: {len(train_ds)} | Val: {len(val_ds)}')
    return train_loader, val_loader

print('Dataset & augmentation loaded.')

## 🧠 Cell 4 — HSI ViT U-Net Architecture

Key difference from RGB: `SpectralProjection` makes the model band-agnostic (works with 260 or 310 bands). Uses `nn.LazyConv2d` which initialises on first forward pass.

In [ ]:
class SpectralProjection(nn.Module):
    """Any bands → fixed (spectral_out, H, W). Band-agnostic via LazyConv2d."""
    def __init__(self, spectral_out=64):
        super().__init__()
        self.proj = nn.Sequential(
            nn.LazyConv2d(128, kernel_size=1, bias=False),
            nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, spectral_out, kernel_size=1, bias=False),
            nn.BatchNorm2d(spectral_out), nn.GELU())
    def forward(self, x): return self.proj(x)

class PatchEmbed(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_ch=64, hidden=256):
        super().__init__()
        self.n_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_ch, hidden, kernel_size=patch_size, stride=patch_size)
    def forward(self, x):
        x = self.proj(x); B,D,H,W = x.shape
        return x.flatten(2).transpose(1,2), H, W

class MHSA(nn.Module):
    def __init__(self, dim, heads, drop=0.0):
        super().__init__()
        self.heads=heads; self.scale=(dim//heads)**-0.5
        self.qkv=nn.Linear(dim,dim*3,bias=False); self.proj=nn.Linear(dim,dim)
        self.drop=nn.Dropout(drop)
    def forward(self, x):
        B,N,D=x.shape; H=self.heads; hd=D//H
        qkv=self.qkv(x).reshape(B,N,3,H,hd).permute(2,0,3,1,4)
        q,k,v=qkv[0],qkv[1],qkv[2]
        attn=self.drop((q@k.transpose(-2,-1))*self.scale).softmax(-1)
        return self.proj((attn@v).transpose(1,2).reshape(B,N,D))

class TransBlock(nn.Module):
    def __init__(self, dim, heads, mlp_dim, drop=0.1):
        super().__init__()
        self.n1=nn.LayerNorm(dim); self.attn=MHSA(dim,heads,drop)
        self.n2=nn.LayerNorm(dim)
        self.mlp=nn.Sequential(nn.Linear(dim,mlp_dim),nn.GELU(),nn.Dropout(drop),
                                nn.Linear(mlp_dim,dim),nn.Dropout(drop))
    def forward(self, x):
        x = x + self.attn(self.n1(x))
        x = x + self.mlp(self.n2(x))
        return x

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, drop=0.1):
        super().__init__()
        self.b = nn.Sequential(
            nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False),nn.BatchNorm2d(out_ch),nn.GELU(),
            nn.Dropout2d(drop),
            nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False),nn.BatchNorm2d(out_ch),nn.GELU())
    def forward(self, x): return self.b(x)

class HSIViTUNet(nn.Module):
    """HSI ViT U-Net. Input (B, n_bands, 32, 32) → (B, 1, 32, 32) logits."""
    def __init__(self, spectral_out=64, img_size=32, patch_size=4,
                 hidden=256, depth=4, heads=8, mlp_dim=1024, drop=0.1, dec_ch=128):
        super().__init__()
        D=hidden; DC=dec_ch
        self.spec_proj   = SpectralProjection(spectral_out)
        self.patch_embed = PatchEmbed(img_size, patch_size, spectral_out, D)
        n = (img_size // patch_size) ** 2
        self.pos_embed   = nn.Parameter(torch.zeros(1, n, D))
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        self.pos_drop    = nn.Dropout(drop)
        self.blocks      = nn.ModuleList([TransBlock(D,heads,mlp_dim,drop) for _ in range(depth)])
        self.norm        = nn.LayerNorm(D)
        self.skip_idx    = {depth//2, depth}
        self.skip_proj   = nn.ModuleList([nn.Conv2d(D,DC,1) for _ in range(2)])
        self.dec2 = ConvBlock(DC+DC, DC,    drop)
        self.dec1 = ConvBlock(DC+DC, DC//2, drop)
        self.out  = nn.Conv2d(DC//2, 1, 1)

    def forward(self, x):
        target = x.shape[2:]
        x = self.spec_proj(x)                             # (B,64,32,32)
        tokens, H, W = self.patch_embed(x)                # (B,64,256)
        tokens = self.pos_drop(tokens + self.pos_embed)
        skips = []
        for i, blk in enumerate(self.blocks, 1):
            tokens = blk(tokens)
            if i in self.skip_idx:
                B,N,D = tokens.shape
                skips.append(tokens.transpose(1,2).reshape(B,D,H,W))
        tokens = self.norm(tokens)
        B,N,D = tokens.shape
        enc = tokens.transpose(1,2).reshape(B,D,H,W)     # (B,D,8,8)
        ep  = self.skip_proj[1](enc)
        s1  = self.skip_proj[0](skips[0])
        s2  = self.skip_proj[1](skips[1])
        d2 = self.dec2(torch.cat([F.interpolate(ep, scale_factor=2, mode='bilinear', align_corners=False), s2], 1))
        d1 = self.dec1(torch.cat([F.interpolate(d2, scale_factor=2, mode='bilinear', align_corners=False), s1], 1))
        return self.out(F.interpolate(d1, size=target, mode='bilinear', align_corners=False))

# Sanity check — test band-agnostic behaviour
for nb in [260, 310]:
    _m = HSIViTUNet(spectral_out=CFG['spectral_out'], img_size=CFG['img_size'],
                    patch_size=CFG['patch_size'], hidden=CFG['vit_hidden'],
                    depth=CFG['vit_depth'], heads=CFG['vit_heads'],
                    mlp_dim=CFG['vit_mlp_dim']).to(DEVICE)
    _x = torch.randn(2, nb, 32, 32).to(DEVICE)
    _o = _m(_x)
    print(f'Bands={nb}  Input:{_x.shape} → Output:{_o.shape}')
print(f'Params: {sum(p.numel() for p in _m.parameters()):,}')
del _m, _x, _o

## ⚖️ Cell 5 — Loss & Metrics

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=1.0):
        super().__init__()
        self.gamma=gamma; self.pw=torch.tensor([pos_weight])
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=self.pw.to(logits.device))
        return ((1-torch.exp(-bce))**self.gamma * bce).mean()

class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-6): super().__init__(); self.s=smooth
    def forward(self, logits, targets):
        p=torch.sigmoid(logits).view(-1); t=targets.view(-1)
        return 1-(2*(p*t).sum()+self.s)/(p.sum()+t.sum()+self.s)

class CombinedLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=1.0, dice_weight=0.5):
        super().__init__()
        self.focal=FocalLoss(gamma,pos_weight); self.dice=DiceLoss(); self.dw=dice_weight
    def forward(self, l, t): return (1-self.dw)*self.focal(l,t)+self.dw*self.dice(l,t)

def compute_iou(preds, targets, thr=0.5, eps=1e-6):
    if targets.sum()==0 and (torch.sigmoid(preds)>thr).sum()==0: return float('nan')
    p=(torch.sigmoid(preds)>thr).float(); i=(p*targets).sum(); u=p.sum()+targets.sum()-i
    return ((i+eps)/(u+eps)).item()

def compute_dice(preds, targets, thr=0.5, eps=1e-6):
    if targets.sum()==0 and (torch.sigmoid(preds)>thr).sum()==0: return float('nan')
    p=(torch.sigmoid(preds)>thr).float(); i=(p*targets).sum()
    return ((2*i+eps)/(p.sum()+targets.sum()+eps)).item()

print('Loss & metrics ready.')

## 🚂 Cell 6 — Training & Validation Loops

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss,total_iou,total_dice,n = 0.,0.,0.,0
    for cubes, masks in tqdm(loader, desc='  Train', leave=False):
        cubes=cubes.to(device,non_blocking=True); masks=masks.to(device,non_blocking=True)
        optimizer.zero_grad()
        logits=model(cubes); loss=criterion(logits,masks)
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(),1.0); optimizer.step()
        total_loss += loss.item()
        iou = compute_iou(logits.detach(), masks)
        if not np.isnan(iou): total_iou+=iou; total_dice+=compute_dice(logits.detach(),masks); n+=1
    N=len(loader); return total_loss/N, total_iou/max(n,1), total_dice/max(n,1)

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    total_loss,total_iou,total_dice,n = 0.,0.,0.,0
    for cubes, masks in tqdm(loader, desc='  Val  ', leave=False):
        cubes=cubes.to(device,non_blocking=True); masks=masks.to(device,non_blocking=True)
        logits=model(cubes); loss=criterion(logits,masks)
        total_loss += loss.item()
        iou = compute_iou(logits, masks)
        if not np.isnan(iou): total_iou+=iou; total_dice+=compute_dice(logits,masks); n+=1
    N=len(loader); return total_loss/N, total_iou/max(n,1), total_dice/max(n,1)

print('Loops ready.')

## 🏋️ Cell 7 — Run Training

⚠️ The dummy forward pass is **required** to initialise `LazyConv2d` in SpectralProjection. Do not skip it.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

train_loader, val_loader, n_bands = build_hsi_dataloaders(CFG)

model = HSIViTUNet(
    spectral_out=CFG['spectral_out'], img_size=CFG['img_size'],
    patch_size=CFG['patch_size'],     hidden=CFG['vit_hidden'],
    depth=CFG['vit_depth'],           heads=CFG['vit_heads'],
    mlp_dim=CFG['vit_mlp_dim'],       drop=CFG['dropout'],
).to(DEVICE)

# Required: initialise LazyConv2d
with torch.no_grad():
    _d = torch.randn(2, n_bands, CFG['img_size'], CFG['img_size']).to(DEVICE)
    _ = model(_d); del _d
print(f'Model initialised | n_bands={n_bands}')
CFG['n_bands_clean'] = n_bands
wandb.config.update({'n_bands_clean': n_bands}, allow_val_change=True)

criterion = CombinedLoss(CFG['focal_gamma'], CFG['pos_weight'], CFG['dice_weight'])
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
warmup    = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=CFG['warmup_epochs'])
cosine    = CosineAnnealingLR(optimizer, T_max=CFG['epochs']-CFG['warmup_epochs'], eta_min=1e-6)
scheduler = SequentialLR(optimizer, schedulers=[warmup,cosine], milestones=[CFG['warmup_epochs']])

wl_clean = np.linspace(400, 2450, n_bands)
nir_idx  = np.argmin(np.abs(wl_clean-850))
red_idx  = np.argmin(np.abs(wl_clean-670))
grn_idx  = np.argmin(np.abs(wl_clean-550))

history = {k:[] for k in ['train_loss','val_loss','train_iou','val_iou',
                            'train_dice','val_dice','train_f1','val_f1']}
best_iou, patience_cnt, PATIENCE = 0.0, 0, 20

print(f'Starting HSI ViT U-Net training for {CFG["epochs"]} epochs...\n')

for epoch in range(1, CFG['epochs']+1):
    tr_loss,tr_iou,tr_dice = train_one_epoch(model,train_loader,optimizer,criterion,DEVICE)
    va_loss,va_iou,va_dice = validate(model,val_loader,criterion,DEVICE)
    tr_f1,va_f1 = tr_dice, va_dice
    scheduler.step()

    for k,v in zip(['train_loss','val_loss','train_iou','val_iou','train_dice','val_dice','train_f1','val_f1'],
                   [tr_loss,va_loss,tr_iou,va_iou,tr_dice,va_dice,tr_f1,va_f1]):
        history[k].append(v)

    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch:03d}/{CFG["epochs"]}]  Loss:{tr_loss:.4f}/{va_loss:.4f}  '
          f'IoU:{tr_iou:.3f}/{va_iou:.3f}  Dice/F1:{tr_dice:.3f}/{va_dice:.3f}  LR:{lr_now:.2e}')
    wandb.log({'epoch':epoch,'lr':lr_now,
               'train/loss':tr_loss,'train/iou':tr_iou,'train/dice':tr_dice,'train/f1':tr_f1,
               'val/loss':va_loss,'val/iou':va_iou,'val/dice':va_dice,'val/f1':va_f1})

    if epoch % 10 == 0:
        model.eval()
        cubes, msks = next(iter(val_loader))
        with torch.no_grad():
            lgts = model(cubes.to(DEVICE))
            prds = (torch.sigmoid(lgts)>0.5).float()
        wim = []
        for i in range(min(4,len(cubes))):
            c = cubes[i].cpu().numpy()
            fc = np.stack([c[nir_idx],c[red_idx],c[grn_idx]],axis=-1)
            fc = (fc-fc.min())/(fc.max()-fc.min()+1e-6)
            wim.append(wandb.Image(fc,
                masks={'ground_truth':{'mask_data':msks[i,0].cpu().numpy(),'class_labels':{0:'bg',1:'tree'}},
                       'prediction':  {'mask_data':prds[i,0].cpu().numpy(),'class_labels':{0:'bg',1:'tree'}}},
                caption=f'Epoch {epoch} | IoU={va_iou:.3f} F1={va_f1:.3f}'))
        wandb.log({'val/predictions':wim})
        model.train()

    if va_iou > best_iou:
        best_iou=va_iou; patience_cnt=0
        ckpt=os.path.join(CFG['save_dir'],'best_model.pth')
        torch.save({'epoch':epoch,'model_state':model.state_dict(),
                    'optim_state':optimizer.state_dict(),
                    'val_iou':best_iou,'val_f1':va_f1,
                    'n_bands':n_bands,'cfg':CFG,'arch':'HSIViTUNet'}, ckpt)
        print(f'  ⭐ Best IoU:{best_iou:.4f}  F1:{va_f1:.4f}')
        wandb.run.summary.update({'best_val_iou':best_iou,'best_val_f1':va_f1,'best_epoch':epoch})
    else:
        patience_cnt+=1
        if patience_cnt>=PATIENCE: print(f'Early stopping at epoch {epoch}'); break

print(f'\nDone! Best val IoU: {best_iou:.4f}')
wandb.finish()

## 📈 Cell 8 — Training Curves

In [ ]:
x = range(1, len(history['train_loss'])+1)
fig, axes = plt.subplots(1,4,figsize=(20,4))
for ax,(key,title) in zip(axes,[('loss','Loss'),('iou','IoU'),('dice','Dice'),('f1','F1')]):
    ax.plot(x, history[f'train_{key}'], label='Train', color='steelblue')
    ax.plot(x, history[f'val_{key}'],   label='Val',   color='tomato')
    ax.set_title(title,fontsize=13); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle('HSI ViT U-Net — Tree Crown Segmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('hsi_vit_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 📊 Cell 9 — Compare vs HSI 3D CNN Baseline

Fill in your 3D CNN baseline numbers.

In [ ]:
baseline_iou  = 0.0   # <- update with HSI 3D CNN best val IoU
baseline_f1   = 0.0   # <- update with HSI 3D CNN best val F1
baseline_name = 'HSI 3D CNN'

vit_iou = best_iou
vit_f1  = history['val_f1'][history['val_iou'].index(best_iou)]

models=[baseline_name,'HSI ViT U-Net']; ious=[baseline_iou,vit_iou]; f1s=[baseline_f1,vit_f1]
fig,axes = plt.subplots(1,2,figsize=(10,4)); colors=['steelblue','mediumseagreen']
for ax,vals,title in zip(axes,[ious,f1s],['Validation IoU','Validation F1/Dice']):
    ax.bar(models,vals,color=colors,width=0.4)
    ax.set_title(title,fontsize=13); ax.set_ylim(0,1); ax.grid(axis='y',alpha=0.3)
    for i,v in enumerate(vals): ax.text(i,v+0.01,f'{v:.3f}',ha='center',fontsize=11)
plt.suptitle('Model Comparison — HSI Tree Crown',fontsize=14,fontweight='bold')
plt.tight_layout(); plt.savefig('hsi_model_comparison.png',dpi=120,bbox_inches='tight'); plt.show()

winner = 'HSI ViT U-Net' if vit_iou>baseline_iou else baseline_name
print(f'Winner: {winner}  (delta IoU={abs(vit_iou-baseline_iou):.3f})')
print(f'\nModel                IoU    F1')
for m,i,f in zip(models,ious,f1s): print(f'{m:<20} {i:.3f}  {f:.3f}')